Đoạn code này lấy api của Youtube để tìm kiếm các video với từ khóa khác biệt


In [ ]:
from googleapiclient.discovery import build
import pandas as pd
from dotenv import load_dotenv
import os
import time

# Đọc API Key từ file .env
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build('youtube', 'v3', developerKey=API_KEY)

def get_youtube_data_expanded(queries, max_pages=3):
    all_data = []
    for q in queries:
        print(f"\n🔍 Đang lấy dữ liệu cho: '{q}'...")
        next_page_token = None
        page_count = 0

        while page_count < max_pages:
            try:
                # 1. Tìm kiếm video (có pagination)
                search_params = {
                    'q': q,
                    'part': 'snippet',
                    'maxResults': 50,
                    'type': 'video',
                    'order': 'relevance',
                }
                if next_page_token:
                    search_params['pageToken'] = next_page_token

                search_response = youtube.search().list(**search_params).execute()

                video_ids = [item['id']['videoId'] for item in search_response.get('items', [])]

                if not video_ids:
                    print(f"   ⚠ Không tìm thấy thêm video.")
                    break

                # 2. Lấy chi tiết thống kê
                video_response = youtube.videos().list(
                    part='snippet,statistics',
                    id=','.join(video_ids)
                ).execute()

                for item in video_response.get('items', []):
                    all_data.append({
                        'Query': q,
                        'Video_ID': item['id'],
                        'Published_At': item['snippet']['publishedAt'],
                        'Channel_Title': item['snippet']['channelTitle'],
                        'Title': item['snippet']['title'],
                        'Description': item['snippet']['description'],
                        'Tags': ",".join(item['snippet'].get('tags', [])),
                        'Category_ID': item['snippet'].get('categoryId', ''),
                        'View_Count': int(item['statistics'].get('viewCount', 0)),
                        'Like_Count': int(item['statistics'].get('likeCount', 0)),
                        'Comment_Count': int(item['statistics'].get('commentCount', 0)),
                        'Favorite_Count': int(item['statistics'].get('favoriteCount', 0)),
                    })

                page_count += 1
                next_page_token = search_response.get('nextPageToken')

                print(f"   ✅ Trang {page_count}: lấy được {len(video_response.get('items', []))} video (tổng: {len(all_data)})")

                if not next_page_token:
                    break

                # Nghỉ 1 giây để tránh bị rate limit
                time.sleep(1)

            except Exception as e:
                print(f"   ❌ Lỗi: {e}")
                # Nếu bị quota limit thì dừng
                if "quotaExceeded" in str(e) or "rateLimitExceeded" in str(e):
                    print("\n⚠️  ĐÃ HẾT QUOTA API! Lưu dữ liệu đã lấy được...")
                    break
                time.sleep(2)
                break

        # Nghỉ giữa mỗi keyword
        time.sleep(0.5)

    return pd.DataFrame(all_data)



keywords = [
    # --- Xu hướng chung ---
    "flower arrangement trends 2024",
    "flower arrangement trends 2025",
    "wedding flower trends 2025",
    "popular flowers for gift",
    "florist business trends",
    "floral design ideas 2025",

    # --- Từng loại hoa cụ thể ---
    "rose bouquet trending",
    "tulip arrangement ideas",
    "orchid decoration trends",
    "sunflower bouquet popular",
    "peony wedding flowers",
    "hydrangea arrangement trending",
    "lily flower gift ideas",
    "dahlia floral design",
    "lavender bouquet trending",
    "chrysanthemum decoration ideas",
    "ranunculus wedding bouquet",
    "carnation flower arrangement",

    # --- Theo sự kiện ---
    "valentine flowers trending",
    "wedding bouquet ideas 2025",
    "birthday flower gift ideas",
    "funeral flowers arrangement",
    "christmas flower decoration",
    "mother's day flower trends",
    "graduation flower bouquet",

    # --- Theo phong cách ---
    "minimalist flower arrangement",
    "luxury floral design",
    "dried flower arrangement trending",
    "sustainable floristry trends",
    "boho wedding flowers",

    # --- Business & Market ---
    "flower shop business tips",
    "floral industry trends 2025",
    "best selling flowers 2024",
    "flower market demand",
]

if __name__ == "__main__":
    print("=" * 60)
    print("  YOUTUBE DATA COLLECTION - EXPANDED VERSION")
    print(f"  Số keywords: {len(keywords)}")
    print(f"  Max pages/keyword: 2 (tối đa ~100 video/keyword)")
    print(f"  Ước tính: {len(keywords) * 100} video tối đa")
    print("=" * 60)

    # Lấy 2 trang mỗi keyword (khoảng 100 video/keyword)
    # Nếu muốn ít hơn để tiết kiệm quota, đặt max_pages=1
    df = get_youtube_data_expanded(keywords, max_pages=2)

    # Loại bỏ video trùng lặp (cùng Video_ID)
    before = len(df)
    df = df.drop_duplicates(subset='Video_ID', keep='first')
    after = len(df)
    print(f"\n📊 Loại bỏ {before - after} video trùng lặp")

    # Lưu file
    df.to_csv("flower_social_data_expanded.csv", index=False, encoding='utf-8-sig')

    print(f"\n{'=' * 60}")
    print(f"  ✅ HOÀN THÀNH!")
    print(f"  📁 Đã lưu {len(df)} dòng dữ liệu")
    print(f"  📁 File: flower_social_data_expanded.csv")
    print(f"  📁 Keywords đã quét: {df['Query'].nunique()}")
    print(f"{'=' * 60}")

    # Thống kê nhanh
    print(f"\n📈 Thống kê:")
    print(f"  - Tổng video: {len(df)}")
    print(f"  - Tổng views: {df['View_Count'].sum():,.0f}")
    print(f"  - Trung bình views: {df['View_Count'].mean():,.0f}")
    print(f"  - Video nhiều views nhất: {df['View_Count'].max():,.0f}")
    print(f"\n  Số video theo keyword:")
    print(df['Query'].value_counts().to_string())


In [ ]:
import pandas as pd
from google.colab import drive

# Bước 1: Cấp quyền và kết nối (mount) Google Drive với Colab
# Khi chạy dòng này, Colab sẽ yêu cầu bạn cấp quyền truy cập Drive
drive.mount('/content/drive')

# Bước 2: Khai báo đường dẫn.
# Mặc định Google Drive của bạn sẽ nằm ở đường dẫn: /content/drive/MyDrive/
folder_path = '/content/drive/MyDrive/Data_BDM/'

youtube_path = folder_path + 'Youtube_data.csv'
instagram_path = folder_path + 'Instagram_data.csv'

# Bước 3: Đọc dữ liệu vào DataFrame
print("Đang tải dữ liệu, vui lòng chờ...")

try:
    # Đọc file YouTube
    df_youtube = pd.read_csv(youtube_path)
    print("✅ Đã đọc thành công file Youtube_data.csv!")
    print(f" - Số dòng: {df_youtube.shape[0]} | Số cột: {df_youtube.shape[1]}")

    # Đọc file Instagram
    # (Thêm low_memory=False vì file này có số lượng cột rất lớn và kiểu dữ liệu hỗn hợp)
    df_instagram = pd.read_csv(instagram_path, low_memory=False)
    print("✅ Đã đọc thành công file Instagram_data.csv!")
    print(f" - Số dòng: {df_instagram.shape[0]} | Số cột: {df_instagram.shape[1]}\n")

except FileNotFoundError as e:
    print(f"❌ Lỗi: Không tìm thấy file. Vui lòng kiểm tra lại đường dẫn!\nChi tiết lỗi: {e}")

# Xem thử 3 dòng đầu của mỗi data để kiểm tra
print("--- Dữ liệu Youtube (3 dòng đầu) ---")
display(df_youtube.head(10))

print("--- Dữ liệu Instagram (3 dòng đầu) ---")
display(df_instagram.head(10))

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install emoji pandas

In [ ]:
2.Clean data

SyntaxError: invalid decimal literal (547808722.py, line 1)

In [ ]:
import pandas as pd
import re
import emoji
import unicodedata

# 1. HÀM DỌN DẸP TEXT CHUNG
def clean_caption_text(text):
    if pd.isna(text):
        return text
    text = str(text)

    # Chuẩn hóa font chữ kiểu
    text = unicodedata.normalize('NFKC', text)

    # Cắt bỏ phần sau các đường kẻ phân cách dài
    text = re.split(r'[-=_]{4,}', text)[0]

    # Quét từ khóa liên hệ và cắt bỏ
    contact_keywords = r'(?i)(hotline|add\s*:|địa chỉ\s*:|zalo|z\.lo|sđt|điện thoại|ig\s*:|instagram|liên hệ|inbox|tiệm hoa|chi nhánh|cs\d:|cơ sở|website|shopee)'
    match = re.search(contact_keywords, text)
    if match:
        text = text[:match.start()]

    # Dọn dẹp emoji, hashtag, khoảng trắng...
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'(?:0|\+84)(?:\s|\.|-)?\d{2,3}(?:\s|\.|-)?\d{3}(?:\s|\.|-)?\d{3,4}', '', text)
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()

def extract_hashtags(text):
    if pd.isna(text):
        return ""
    return " ".join(re.findall(r'#\S+', str(text)))

# 2. HÀM CLEAN YOUTUBE
def clean_youtube_df(df):
    df_clean = df.copy()
    keep_cols = ['Channel_Title', 'Title', 'Description', 'Tags', 'View_Count', 'Like_Count', 'Published_At']
    actual_keep = [c for c in keep_cols if c in df_clean.columns]
    df_clean = df_clean[actual_keep]

    print("Đang xử lý Youtube...")
    extracted_tags = pd.Series("", index=df_clean.index)

    if 'Title' in df_clean.columns:
        extracted_tags += " " + df_clean['Title'].apply(extract_hashtags)
    if 'Description' in df_clean.columns:
        extracted_tags += " " + df_clean['Description'].apply(extract_hashtags)

    if 'Tags' in df_clean.columns:
        df_clean['Tags'] = df_clean['Tags'].fillna("") + extracted_tags
        df_clean['Tags'] = df_clean['Tags'].str.replace(r'\s+', ' ', regex=True).str.strip()
    else:
        df_clean['Tags'] = extracted_tags.str.strip()

    if 'Title' in df_clean.columns:
        df_clean['Title'] = df_clean['Title'].apply(clean_caption_text)
    if 'Description' in df_clean.columns:
        df_clean['Description'] = df_clean['Description'].apply(clean_caption_text)

    return df_clean

# 3. HÀM CLEAN INSTAGRAM
def clean_instagram_df(df):
    df_clean = df.copy()
    base_keep = ['caption', 'likesCount', 'commentsCount', 'ownerUsername']
    hashtag_cols = [f'hashtags/{i}' for i in range(13)]
    keep_cols = base_keep + hashtag_cols

    actual_keep = [c for c in keep_cols if c in df_clean.columns]
    df_clean = df_clean[actual_keep]

    print("Đang xử lý Instagram...")
    hashtag_cols_actual = [c for c in hashtag_cols if c in df_clean.columns]
    if hashtag_cols_actual:
        df_clean['Tags'] = df_clean[hashtag_cols_actual].apply(
            lambda row: ' '.join(['#' + str(val) for val in row.dropna() if str(val).strip() != '']),
            axis=1
        )
        df_clean = df_clean.drop(columns=hashtag_cols_actual)

    if 'caption' in df_clean.columns:
        df_clean['caption'] = df_clean['caption'].apply(clean_caption_text)

    return df_clean

# --- THỰC THI GỌI LẠI BIẾN CŨ TỪ CELL TRƯỚC ---

# Dùng luôn biến df_youtube và df_instagram đã có trên RAM!
df_youtube_cleaned = clean_youtube_df(df_youtube)
df_instagram_cleaned = clean_instagram_df(df_instagram)

print("\n✅ HOÀN TẤT CLEAN DATA!")

# Xem thử kết quả
print("\n--- Dữ liệu Youtube (Đã dọn dẹp) ---")
display(df_youtube_cleaned.head(3))

print("\n--- Dữ liệu Instagram (Đã dọn dẹp) ---")
display(df_instagram_cleaned.head(3))

ModuleNotFoundError: No module named 'emoji'

In [ ]:
# 1. Dùng .shape để in ra nhanh (Số dòng, Số cột)
print("Kích thước Youtube:", df_youtube_cleaned.shape)
print("Kích thước Instagram:", df_instagram_cleaned.shape)

# Nếu muốn in ra câu chữ cho dễ đọc hơn:
print(f"Youtube có {df_youtube_cleaned.shape[0]} dòng và {df_youtube_cleaned.shape[1]} cột.")
print(f"Instagram có {df_instagram_cleaned.shape[0]} dòng và {df_instagram_cleaned.shape[1]} cột.")